# Qwen3.5-4B + LoRA: 300 questions

The first 19 answers were extracted at `<|im_end|>` after a stop-token mismatch.


In [ ]:
%pip uninstall -y -q torchao
%pip install -q "torch==2.11.0" "transformers==5.22.1" "accelerate==1.15.0" "peft==0.21.2" "tokenizers==0.23.3" "safetensors==0.8.0" "huggingface-hub==1.33.0"

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import json
import os
from pathlib import Path
import subprocess
import sys
import time
import traceback
import zipfile

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import torch
from transformers import (
    AutoTokenizer,
    LogitsProcessor,
    LogitsProcessorList,
    Qwen3_5ForConditionalGeneration,
    set_seed,
)

run_name = "qwen35_4b_lora_nonthinking"
run_dir = Path("/content/data/runs/plant_care_pilot") / run_name
dataset_path = Path("/content/data/prepared/plant_care_pilot/questions.jsonl")
baseline_dir = Path("/content/data/runs/plant_care_pilot/qwen35_4b_nonthinking")
adapter_dir = Path("/content/data/runs/plant_care_train_1000/qwen35_4b_lora/adapter")
config = json.loads((baseline_dir / "config.json").read_text())
config["eos_token_id"] = 248046
ready_for_generation = False
print(run_name)

qwen35_4b_lora_nonthinking


In [ ]:
if not dataset_path.exists():
    raise FileNotFoundError("Upload questions.jsonl before running this cell.")

dataset_bytes = dataset_path.read_bytes()
questions = [json.loads(line) for line in dataset_bytes.decode().splitlines() if line.strip()]
if len(questions) != 300 or len({row["id"] for row in questions}) != 300:
    raise ValueError("Expected 300 unique questions.")
seed_by_id = {row["id"]: config["seed"] + i for i, row in enumerate(questions, 1)}
if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("This notebook requires an NVIDIA T4.")

run_dir.mkdir(parents=True, exist_ok=True)
config_path = run_dir / "config.json"
if config_path.exists() and json.loads(config_path.read_text()) != config:
    raise ValueError("This run already has different settings. Choose a new run name.")
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n")

environment = {
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_gib": torch.cuda.get_device_properties(0).total_memory / 2**30,
    "compute_capability": list(torch.cuda.get_device_capability(0)),
    "cuda": torch.version.cuda,
    "packages": {name: version(name) for name in [
        "torch", "transformers", "accelerate", "tokenizers", "safetensors", "huggingface-hub", "peft"
    ]},
    "nvidia_smi": subprocess.check_output(["nvidia-smi"], text=True),
}
(run_dir / "environment.json").write_text(
    json.dumps(environment, ensure_ascii=False, indent=2) + "\n"
)
(run_dir / "requirements-freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
)
print(json.dumps(environment, ensure_ascii=False, indent=2))

{
  "started_at_utc": "2026-11-04T11:12:20.879480+00:00",
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "gpu": "Tesla T4",
  "gpu_memory_gib": 14.56317138671875,
  "compute_capability": [
    7,
    5
  ],
  "cuda": "13.0",
  "packages": {
    "torch": "2.11.0+cu130",
    "transformers": "5.22.1",
    "accelerate": "1.15.0",
    "tokenizers": "0.23.3",
    "safetensors": "0.8.0",
    "huggingface-hub": "1.33.0",
    "peft": "0.21.2"
  },
  "nvidia_smi": "Wed Nov  4 11:12:20 2026       \n+-----------------------------------------------------------------------------------------+\n| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |\n+-----------------------------------------+------------------------+----------------------+\n| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |\n| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |\n|                    

In [ ]:
load_started = time.perf_counter()
torch.cuda.reset_peak_memory_stats()
tokenizer = AutoTokenizer.from_pretrained(
    config["model_id"], revision=config["model_revision"], token=False
)
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    config["model_id"],
    revision=config["model_revision"],
    dtype=torch.float16,
    device_map={"": 0},
    attn_implementation=config["attention_implementation"],
    use_safetensors=True,
    token=False,
)
from peft import PeftModel, get_peft_model_state_dict
from safetensors.torch import load_file

model = PeftModel.from_pretrained(
    model, str(adapter_dir), is_trainable=False,
    key_mapping={r"^model\.layers\.": "model.language_model.layers."},
)
saved = load_file(str(adapter_dir / "adapter_model.safetensors"), device="cpu")
loaded = {key.replace(".model.language_model.layers.", ".model.layers."): value for key, value in get_peft_model_state_dict(model).items()}
assert loaded.keys() == saved.keys()
assert all(torch.equal(loaded[key].detach().cpu(), saved[key]) for key in saved)
verified_adapter_tensors = len(saved)
del saved, loaded
model.generation_config.eos_token_id = config["eos_token_id"]
model.eval()
references = {row["id"]: row for row in (json.loads(line) for line in (baseline_dir / "answers.jsonl").read_text().splitlines())}
for row in questions:
    messages = [{"role": "system", "content": config["system_prompt"]}, {"role": "user", "content": row["question"]}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    assert prompt == references[row["id"]]["rendered_prompt"]
    assert seed_by_id[row["id"]] == references[row["id"]]["generation_seed"]
    assert len(tokenizer(prompt, add_special_tokens=False)["input_ids"]) == references[row["id"]]["input_tokens"]
load_stats = {
    "load_seconds": time.perf_counter() - load_started,
    "allocated_gib": torch.cuda.memory_allocated() / 2**30,
    "peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
    "model_class": type(model).__name__,
    "base_model_class": type(model.get_base_model()).__name__,
    "verified_adapter_tensors": verified_adapter_tensors,
    "verified_baseline_prompts": len(questions),
}
(run_dir / "model-loading.json").write_text(json.dumps(load_stats, indent=2) + "\n")
print(json.dumps(load_stats, indent=2))
ready_for_generation = True

{
  "load_seconds": 39.71228567600019,
  "allocated_gib": 8.534170150756836,
  "peak_reserved_gib": 8.625,
  "model_class": "PeftModelForCausalLM",
  "base_model_class": "Qwen3_5ForConditionalGeneration",
  "verified_adapter_tensors": 256,
  "verified_baseline_prompts": 300
}


In [ ]:
class PresencePenalty(LogitsProcessor):
    def __init__(self, penalty, prompt_length):
        self.penalty = penalty
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores):
        generated = input_ids[:, self.prompt_length:]
        if generated.numel():
            seen = torch.zeros_like(scores, dtype=torch.bool)
            seen.scatter_(1, generated, True)
            scores = scores - seen.to(scores.dtype) * self.penalty
        return scores


answers_path = run_dir / "answers.jsonl"
answers = [json.loads(line) for line in answers_path.read_text().splitlines()] if answers_path.exists() else []
if len({row["id"] for row in answers}) != len(answers):
    raise ValueError("Duplicate question IDs in saved answers.")


def save_artifacts():
    ordered = sorted(answers, key=lambda row: row["id"])
    summary = {
        "run_name": run_name,
        "completed": len(ordered),
        "expected": len(questions),
        "output_tokens": sum(row["output_tokens"] for row in ordered),
        "generation_seconds": sum(row["elapsed_seconds"] for row in ordered),
        "truncated_ids": [row["id"] for row in ordered if row["finish_reason"] == "length"],
        "answer_truncated_ids": [row["id"] for row in ordered if row.get("answer_end_reason", row["finish_reason"]) == "length"],
        "recovered_initial_answers": sum(row.get("extracted_at_im_end", False) for row in ordered),
        "thinking_marker_ids": [row["id"] for row in ordered if row["thinking_markers"]],
        "peak_reserved_gib": max([load_stats["peak_reserved_gib"]] + [row["peak_reserved_gib"] for row in ordered]),
    }
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    sections = ["# Qwen3.5-4B + LoRA answers", f"Run: `{run_name}`. Non-thinking mode."]
    for row in ordered:
        sections.extend([f"## Question {row['id']}", row["question"], row["answer"]])
    (run_dir / "answers.md").write_text("\n\n".join(sections) + "\n")
    with zipfile.ZipFile(run_dir / "artifacts.zip", "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(run_dir.iterdir()):
            if path.is_file() and path.name != "artifacts.zip":
                archive.write(path, arcname=path.name)


def answer_question(row):
    if row["id"] in {answer["id"] for answer in answers}:
        return
    seed = seed_by_id[row["id"]]
    set_seed(seed)
    messages = [
        {"role": "system", "content": config["system_prompt"]},
        {"role": "user", "content": row["question"]},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda:0")
    input_length = inputs["input_ids"].shape[1]
    if input_length > config["max_input_tokens"]:
        raise ValueError(f"Question {row['id']} exceeds the input token limit.")
    penalty = PresencePenalty(config["presence_penalty"], input_length)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            **config["generation"],
            logits_processor=LogitsProcessorList([penalty]),
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    generated = output[0, input_length:]
    raw_response = tokenizer.decode(generated, skip_special_tokens=False)
    eos_ids = model.generation_config.eos_token_id
    eos_ids = eos_ids if isinstance(eos_ids, list) else [eos_ids]
    finished = bool(generated.numel() and generated[-1].item() in eos_ids)
    answer = {
        "id": row["id"],
        "question": row["question"],
        "messages": messages,
        "rendered_prompt": prompt,
        "answer": tokenizer.decode(generated, skip_special_tokens=True).strip(),
        "raw_response": raw_response,
        "generation_seed": seed,
        "input_tokens": input_length,
        "output_tokens": generated.numel(),
        "elapsed_seconds": elapsed,
        "tokens_per_second": generated.numel() / elapsed,
        "finish_reason": "eos" if finished else "length",
        "thinking_markers": "<think>" in raw_response or "</think>" in raw_response,
        "peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
    }
    with answers_path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(answer, ensure_ascii=False) + "\n")
    answers.append(answer)
    save_artifacts()
    print(f"{len(answers)}/{len(questions)} {row['id']}: {answer['output_tokens']} tokens, {elapsed:.1f} s, {answer['finish_reason']}", flush=True)


In [ ]:
assert ready_for_generation
try:
    for row in questions:
        answer_question(row)
except Exception:
    (run_dir / "error.txt").write_text(traceback.format_exc())
    raise
finally:
    save_artifacts()

print((run_dir / "summary.json").read_text())

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


20/300 pcp-garden-020: 140 tokens, 15.9 s, eos


21/300 pcp-garden-021: 140 tokens, 13.5 s, eos


... (276 outputs left out)

298/300 pcp-indoor-148: 150 tokens, 14.3 s, eos


299/300 pcp-indoor-149: 139 tokens, 13.5 s, eos


300/300 pcp-indoor-150: 149 tokens, 14.5 s, eos


{
  "run_name": "qwen35_4b_lora_nonthinking",
  "completed": 300,
  "expected": 300,
  "output_tokens": 46194,
  "generation_seconds": 4458.8667070759875,
  "truncated_ids": [
    "pcp-garden-001",
    "pcp-garden-002",
    "pcp-garden-003",
    "pcp-garden-004",
    "pcp-garden-005",
    "pcp-garden-006",
    "pcp-garden-007",
    "pcp-garden-008",
    "pcp-garden-009",
    "pcp-garden-010",
    "pcp-garden-011",
    "pcp-garden-012",
    "pcp-garden-013",
    "pcp-garden-014",
    "pcp-garden-015",
    "pcp-garden-016",
    "pcp-garden-017",
    "pcp-garden-018",
    "pcp-garden-019"
  ],
  "answer_truncated_ids": [],
  "recovered_initial_answers": 19,
  "thinking_marker_ids": [],
  "peak_reserved_gib": 8.7734375
}

